# ARC-AGI-3 Milestone 2 — Lean baseline

This is the effect-model and graph explorer baseline, not the unreleased learned internal-world agent. No new performance claim is made. Code is embedded below; there are no external private inputs or model weights.

## Method

Each game starts with new task-local transition memory. The agent hashes settled observations, filters actions, suppresses no-op/loop behavior, proposes coordinates from visible components, estimates action effects and traverses observed transition edges toward unexplored frontiers. It does not inspect hidden game source or carry hidden-game experience across games. The budget and stopping rules are defined in MyAgent below.

## Reproduce

Attach only the ARC Prize 2026 ARC-AGI-3 competition input. Select CPU (no GPU or neural weights required), disable internet, and Save Version / Save & Run All from a fresh session. The commit run validates offline runtime and produces an explicitly unscored dummy parquet. Submit this completed notebook version to the competition, selecting submission.parquet; Kaggle's competition rerun runs the actual agent through its gateway and creates the scored parquet. The notebook rejects a one-row dummy in scoring mode. Allow up to 8 hours for the agent subprocess, within Kaggle's 9-hour limit. Record the image versions, logs, version URL, receipt and scored outcome separately.

## License for submitter-authored code and methods

SPDX-License-Identifier: MIT-0

```text
MIT No Attribution

Copyright (c) 2026 J. Thomas Lockhart

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

```

## Third-party provenance and notices

The agent adapts Shlok Sah's MIT-0 FrugalExplorer, revision 7dccef15650fa00ef1db1c21cc7279adcf4a9e2a. Its notice follows:

```text
MIT No Attribution

Copyright (c) 2026 Shlok Sah

Permission is hereby granted, free of charge, to any person obtaining a copy of
this software and associated documentation files (the "Software"), to deal in
the Software without restriction, including without limitation the rights to
use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies
of the Software, and to permit persons to whom the Software is furnished to do
so.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

```

The notebook uses ARC Prize starter plumbing (revision eeb1535404f321d280a8f9194bbc1d7aca5f05fc) and copies the competition-provided framework at runtime. It does not redistribute the competition data. Preserve all upstream notices in the runtime framework. ARC-AGI-3-Agents notice follows:

```text
MIT License

Copyright (c) 2025 ARC Prize

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```

Offline runtime: arc-agi 0.9.8, arcengine 0.9.3, Pillow 12.2.0 from the competition input. pandas, pyarrow and python-dotenv are consumed from Kaggle's image; the runtime probe prints the installed versions. Their upstream licenses remain applicable. No pretrained model, private dataset, persistent PHCG memory, or training corpus is included.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

# Keep ARC's Pillow 12 requirement out of Kaggle's global Python
# environment, where preinstalled Gradio requires Pillow < 12.
# The agent subprocess receives this directory through PYTHONPATH;
# Kaggle's notebook renderer continues using its original packages.
RUNTIME_DIR = '/tmp/arc3_runtime'
WHEEL_DIR = ('/kaggle/input/competitions/'
             'arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
wheel_dir = Path(WHEEL_DIR)
required_wheels = {
    'arc-agi 0.9.8': 'arc_agi-0.9.8-*.whl',
    'arcengine 0.9.3': 'arcengine-0.9.3-*.whl',
    'pillow 12.2.0': 'pillow-12.2.0-*.whl',
}
missing = [
    label
    for label, pattern in required_wheels.items()
    if not any(wheel_dir.glob(pattern))
]
if missing:
    raise FileNotFoundError(
        f'Missing required offline wheels in {WHEEL_DIR}: {missing}'
    )

# A notebook cell can be rerun in the same worker. Rebuild the
# isolated target so stale package files can never survive.
runtime_dir = Path(RUNTIME_DIR)
if runtime_dir.exists():
    shutil.rmtree(runtime_dir)
runtime_dir.mkdir(parents=True)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        '--disable-pip-version-check',
        '--quiet',
        '--no-index', '--find-links', WHEEL_DIR,
        '--target', RUNTIME_DIR, '--no-deps',
        'arc-agi==0.9.8',
        'arcengine==0.9.3',
        'pillow==12.2.0',
    ],
    check=True,
    timeout=300,
)

runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = (
    RUNTIME_DIR + os.pathsep + runtime_env.get('PYTHONPATH', '')
)
runtime_env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
runtime_env['PYTHONDONTWRITEBYTECODE'] = '1'
runtime_env['PYTHONHASHSEED'] = '0'
probe = subprocess.run(
    [
        sys.executable,
        '-X', 'frozen_modules=off',
        '-c',
        ('from importlib.metadata import version; '
         'import arc_agi, arcengine, PIL, dotenv; '
         'print("ARC runtime OK:", '
         'version("arc-agi"), version("arcengine"), '
         'version("pillow"), version("python-dotenv")); '
         'print("ARC runtime path:", arc_agi.__file__)'),
    ],
    env=runtime_env,
    capture_output=True,
    text=True,
    timeout=60,
)
if probe.returncode != 0:
    raise RuntimeError(
        'ARC runtime probe failed:\n'
        + probe.stdout
        + probe.stderr
    )
print(probe.stdout, end='')


In [ ]:
from pathlib import Path
_AGENT_ROOT = Path('/tmp/arc3_baseline')
_AGENT_ROOT.mkdir(parents=True, exist_ok=True)
(_AGENT_ROOT / 'my_agent.py').write_text('"""Lean ARC-AGI-3 effect-model and graph explorer.\n\nThe agent learns only from transitions in the current game. It masks stable\nengine UI bands, snaps coordinate actions to visible components, estimates\nwhich action/color/region combinations produce novel states, and replays known\nedges to return to unexplored frontiers. Confirmed movement actions receive a\nsmall goal-directed bonus toward rare objects.\n\nThe frozen-mask/effect-model/frontier pattern is adapted from Shlok Sah\'s\nMIT-0 FrugalExplorer (commit 7dccef15650fa00ef1db1c21cc7279adcf4a9e2a).\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport math\nimport random\nfrom collections import Counter, defaultdict, deque\nfrom time import perf_counter\nfrom typing import Any\n\nfrom arcengine import FrameData, GameAction, GameState\nfrom agents.agent import Agent\n\n\nGrid = list[list[int]]\nActionKey = tuple[int, int | None, int | None]\n\n\ndef frame_delta(before: Grid, after: Grid) -> dict[str, Any]:\n    """Return a compact, JSON-safe description of an observed frame change."""\n    before_shape = (len(before), len(before[0]) if before else 0)\n    after_shape = (len(after), len(after[0]) if after else 0)\n    if before_shape != after_shape:\n        return {\n            "class": "resize",\n            "changed_cells": None,\n            "bbox": None,\n            "before_shape": list(before_shape),\n            "after_shape": list(after_shape),\n        }\n\n    changed = [\n        (x, y)\n        for y, row in enumerate(after)\n        for x, value in enumerate(row)\n        if value != before[y][x]\n    ]\n    if not changed:\n        delta_class = "unchanged"\n        bbox = None\n    else:\n        xs = [x for x, _ in changed]\n        ys = [y for _, y in changed]\n        bbox = [min(xs), min(ys), max(xs), max(ys)]\n        total = max(1, after_shape[0] * after_shape[1])\n        bbox_area = (bbox[2] - bbox[0] + 1) * (bbox[3] - bbox[1] + 1)\n        if len(changed) <= max(4, math.ceil(total * 0.02)):\n            delta_class = "sparse"\n        elif bbox_area <= total * 0.25:\n            delta_class = "local"\n        else:\n            delta_class = "global"\n    return {\n        "class": delta_class,\n        "changed_cells": len(changed),\n        "bbox": bbox,\n        "before_shape": list(before_shape),\n        "after_shape": list(after_shape),\n    }\n\n\ndef frame_grid(frame: FrameData) -> Grid:\n    if not frame.frame:\n        return []\n    # An action may return a complete animation stack. The last image is the\n    # settled board on which the next action will operate; the first image can\n    # still describe an in-flight state from the previous transition.\n    return [[int(value) for value in row] for row in frame.frame[-1]]\n\n\ndef _masked_grid(grid: Grid) -> Grid:\n    """Mask engine UI bands without changing the observation used for actions."""\n    if not grid or not grid[0]:\n        return []\n    height, width = len(grid), len(grid[0])\n    masked = [row[:] for row in grid]\n    if height < 8 or width < 8:\n        return masked\n    sentinel = 16\n    for y in range(height):\n        for x in range(width):\n            if x in (0, width - 1) or y in (0, 1, height - 1):\n                masked[y][x] = sentinel\n    return masked\n\n\ndef state_key(frame: FrameData) -> str:\n    grid = _masked_grid(frame_grid(frame))\n    height = len(grid)\n    width = len(grid[0]) if height else 0\n    payload = bytearray(\n        (frame.levels_completed & 0xFF, height & 0xFF, width & 0xFF)\n    )\n    for row in grid:\n        payload.extend(value & 0xFF for value in row)\n    return hashlib.blake2b(payload, digest_size=12).hexdigest()\n\n\ndef _background(grid: Grid) -> int:\n    if not grid or not grid[0]:\n        return 0\n    height, width = len(grid), len(grid[0])\n    edge = grid[0] + grid[-1]\n    edge += [grid[y][0] for y in range(1, height - 1)]\n    edge += [grid[y][width - 1] for y in range(1, height - 1)]\n    return Counter(edge).most_common(1)[0][0]\n\n\ndef _components(grid: Grid) -> list[tuple[int, int, int, int, int, int]]:\n    """Return (area, color, cx, cy, min_x, min_y) per color component."""\n    if not grid or not grid[0]:\n        return []\n    height, width = len(grid), len(grid[0])\n    background = _background(grid)\n    x_lo, x_hi = (0, width) if width < 8 else (1, width - 1)\n    y_lo, y_hi = (0, height) if height < 8 else (2, height - 1)\n    seen: set[tuple[int, int]] = set()\n    result: list[tuple[int, int, int, int, int, int]] = []\n    for y in range(y_lo, y_hi):\n        for x in range(x_lo, x_hi):\n            color = grid[y][x]\n            if color == background or (x, y) in seen:\n                continue\n            stack = [(x, y)]\n            seen.add((x, y))\n            cells: list[tuple[int, int]] = []\n            while stack:\n                cx, cy = stack.pop()\n                cells.append((cx, cy))\n                for nx, ny in (\n                    (cx - 1, cy), (cx + 1, cy), (cx, cy - 1), (cx, cy + 1)\n                ):\n                    if (\n                        x_lo <= nx < x_hi\n                        and y_lo <= ny < y_hi\n                        and (nx, ny) not in seen\n                        and grid[ny][nx] == color\n                    ):\n                        seen.add((nx, ny))\n                        stack.append((nx, ny))\n            centre_x = round(sum(x0 for x0, _ in cells) / len(cells))\n            centre_y = round(sum(y0 for _, y0 in cells) / len(cells))\n            result.append(\n                (\n                    len(cells), color, centre_x, centre_y,\n                    min(x0 for x0, _ in cells), min(y0 for _, y0 in cells),\n                )\n            )\n    return sorted(result, key=lambda item: (item[0], item[1], item[3], item[2]))\n\n\ndef coordinate_proposals(grid: Grid, previous: Grid | None = None) -> list[tuple[int, int]]:\n    """Prioritize changed regions, rare components, and real colored pixels."""\n    if not grid or not grid[0]:\n        return [(32, 32)]\n    height, width = len(grid), len(grid[0])\n    x_lo, x_hi = (0, width) if width < 8 else (1, width - 1)\n    y_lo, y_hi = (0, height) if height < 8 else (2, height - 1)\n    proposals: list[tuple[int, int]] = []\n    if previous and len(previous) == height and len(previous[0]) == width:\n        changed = [\n            (x, y)\n            for y in range(y_lo, y_hi)\n            for x in range(x_lo, x_hi)\n            if previous[y][x] != grid[y][x]\n        ]\n        if changed:\n            proposals.append(\n                (\n                    round(sum(x for x, _ in changed) / len(changed)),\n                    round(sum(y for _, y in changed) / len(changed)),\n                )\n            )\n\n    for area, _, cx, cy, min_x, min_y in _components(grid):\n        proposals.append((cx, cy))\n        if area > 3:\n            proposals.append((min_x, min_y))\n\n    background = _background(grid)\n    for y0 in range(y_lo, y_hi, 8):\n        for x0 in range(x_lo, x_hi, 8):\n            found = next(\n                (\n                    (x, y)\n                    for y in range(y0, min(y_hi, y0 + 8))\n                    for x in range(x0, min(x_hi, x0 + 8))\n                    if grid[y][x] != background\n                ),\n                None,\n            )\n            if found:\n                proposals.append(found)\n\n    result: list[tuple[int, int]] = []\n    for x, y in proposals:\n        point = (max(0, min(width - 1, x)), max(0, min(height - 1, y)))\n        if point not in result:\n            result.append(point)\n    return result[:64]\n\n\nclass Node:\n    def __init__(self) -> None:\n        self.candidates: list[ActionKey] = []\n        self.tested: dict[ActionKey, bool] = {}\n        self.edges: dict[ActionKey, str] = {}\n\n\nclass MyAgent(Agent):\n    """Training-free, scoring-aware task-local world-model explorer."""\n\n    MAX_ACTIONS = 400\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        seed = int.from_bytes(\n            hashlib.blake2b(self.game_id.encode("utf-8"), digest_size=8).digest(),\n            "big",\n        )\n        self._rng = random.Random(seed)\n        self._last_state: str | None = None\n        self._last_grid: Grid | None = None\n        self._last_level = 0\n        self._last_action: ActionKey | None = None\n        self._nodes: dict[str, Node] = {}\n        self._state_visits: Counter[str] = Counter()\n        self._states_seen: set[str] = set()\n        self._plan: deque[tuple[str, ActionKey]] = deque()\n        self._no_ops: set[tuple[str, ActionKey]] = set()\n        self._deadly: set[tuple[str, ActionKey]] = set()\n        self._action_stats: dict[int, list[float]] = defaultdict(lambda: [1.0, 2.0])\n        self._color_stats: dict[int, list[float]] = defaultdict(lambda: [1.0, 2.0])\n        self._region_stats: dict[tuple[int, int], list[float]] = defaultdict(\n            lambda: [1.0, 2.0]\n        )\n        self._progress_actions: Counter[int] = Counter()\n        self._move_votes: Counter[tuple[int, int, int, int]] = Counter()\n        self._moves: dict[int, tuple[int, int]] = {}\n        self._avatar_color: int | None = None\n        self._visited_targets: set[tuple[int, int, int]] = set()\n        self._transition_events: deque[dict[str, Any]] = deque(maxlen=512)\n        self._telemetry: Counter[str] = Counter()\n        self._delta_classes: Counter[str] = Counter()\n        self._selection_reasons: Counter[str] = Counter()\n        self._decision_latency_ms_total = 0.0\n        self._decision_latency_ms_max = 0.0\n\n    @property\n    def name(self) -> str:\n        return f"{super().name}.settled-world-model-v2.{self.MAX_ACTIONS}"\n\n    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:\n        return latest_frame.state is GameState.WIN\n\n    def telemetry_summary(self) -> dict[str, Any]:\n        """Summarize task-local behavior without writing from the hot path."""\n        actions = self._telemetry["actions_selected"]\n        transitions = self._telemetry["transitions_observed"]\n        levels = self._telemetry["levels_gained"]\n        return {\n            "schema_version": 1,\n            "game_id": self.game_id,\n            "actions_selected": actions,\n            "transitions_observed": transitions,\n            "unique_states": len(self._states_seen),\n            "current_level_unique_states": len(self._state_visits),\n            "changed_transitions": self._telemetry["changed_transitions"],\n            "novel_transitions": self._telemetry["novel_transitions"],\n            "no_ops": self._telemetry["no_ops"],\n            "no_op_rate": (\n                self._telemetry["no_ops"] / transitions if transitions else 0.0\n            ),\n            "deadly_transitions": self._telemetry["deadly_transitions"],\n            "levels_gained": levels,\n            "actions_per_level": actions / levels if levels else None,\n            "replans": self._telemetry["replans"],\n            "plan_invalidations": self._telemetry["plan_invalidations"],\n            "movement_model": {\n                str(action): [delta[0], delta[1]]\n                for action, delta in sorted(self._moves.items())\n            },\n            "avatar_color": self._avatar_color,\n            "delta_classes": dict(sorted(self._delta_classes.items())),\n            "selection_reasons": dict(sorted(self._selection_reasons.items())),\n            "decision_latency_ms": {\n                "mean": self._decision_latency_ms_total / actions if actions else 0.0,\n                "max": self._decision_latency_ms_max,\n            },\n        }\n\n    def transition_events(self) -> list[dict[str, Any]]:\n        """Return the bounded transition trace for local evaluation tooling."""\n        return list(self._transition_events)\n\n    @staticmethod\n    def _centroid(grid: Grid, color: int) -> tuple[float, float, int] | None:\n        cells = [\n            (x, y)\n            for y in range(2, len(grid) - 1)\n            for x in range(1, len(grid[0]) - 1)\n            if grid[y][x] == color\n        ]\n        if not cells:\n            return None\n        return (\n            sum(x for x, _ in cells) / len(cells),\n            sum(y for _, y in cells) / len(cells),\n            len(cells),\n        )\n\n    def _learn_move(self, action: ActionKey, before: Grid, after: Grid) -> None:\n        if action[1] is not None or not before or not after:\n            return\n        action_id = action[0]\n        colors = set(value for row in before for value in row)\n        colors &= set(value for row in after for value in row)\n        colors.discard(_background(before))\n        colors.discard(0)\n        for color in colors:\n            old = self._centroid(before, color)\n            new = self._centroid(after, color)\n            if old is None or new is None or not (1 <= old[2] <= 40):\n                continue\n            if abs(new[2] - old[2]) > old[2] // 2 + 2:\n                continue\n            dx = round(new[0] - old[0])\n            dy = round(new[1] - old[1])\n            if (dx, dy) == (0, 0) or abs(dx) > 10 or abs(dy) > 10:\n                continue\n            vote = (action_id, dx, dy, color)\n            self._move_votes[vote] += 1\n            if self._move_votes[vote] >= 2:\n                self._moves[action_id] = (dx, dy)\n                self._avatar_color = color\n\n    def _effect_stats(self, action: ActionKey, grid: Grid) -> list[list[float]]:\n        stats = [self._action_stats[action[0]]]\n        if action[1] is not None and action[2] is not None:\n            x, y = action[1], action[2]\n            stats.append(self._color_stats[grid[y][x]])\n            stats.append(self._region_stats[(y // 8, x // 8)])\n        return stats\n\n    def _observe(self, current: str, grid: Grid, level: int, state: GameState) -> None:\n        novel = current not in self._state_visits\n        if self._last_state is not None and self._last_action is not None:\n            edge = (self._last_state, self._last_action)\n            changed = current != self._last_state or level != self._last_level\n            level_delta = max(0, level - self._last_level)\n            delta = frame_delta(\n                _masked_grid(self._last_grid or []), _masked_grid(grid)\n            )\n            self._telemetry["transitions_observed"] += 1\n            self._telemetry["changed_transitions"] += int(changed)\n            self._telemetry["novel_transitions"] += int(changed and novel)\n            self._telemetry["no_ops"] += int(not changed)\n            self._telemetry["deadly_transitions"] += int(\n                state is GameState.GAME_OVER\n            )\n            self._telemetry["levels_gained"] += level_delta\n            self._delta_classes[delta["class"]] += 1\n            self._transition_events.append(\n                {\n                    "step": self._telemetry["transitions_observed"],\n                    "from_state": self._last_state,\n                    "action": {\n                        "id": self._last_action[0],\n                        "x": self._last_action[1],\n                        "y": self._last_action[2],\n                    },\n                    "to_state": current,\n                    "delta": delta,\n                    "novel": bool(changed and novel),\n                    "level_delta": level_delta,\n                    "status": state.name,\n                }\n            )\n            previous_node = self._nodes.get(self._last_state)\n            if previous_node is not None:\n                previous_node.tested[self._last_action] = changed\n                previous_node.edges[self._last_action] = current\n            if not changed:\n                self._no_ops.add(edge)\n            useful = changed and novel\n            for pair in self._effect_stats(self._last_action, self._last_grid or grid):\n                pair[0] += 1.0 if useful else 0.0\n                pair[1] += 1.0\n            if self._last_grid is not None:\n                self._learn_move(self._last_action, self._last_grid, grid)\n            if level > self._last_level:\n                self._progress_actions[self._last_action[0]] += level - self._last_level\n                self._nodes.clear()\n                self._state_visits.clear()\n                self._plan.clear()\n                self._no_ops.clear()\n                self._deadly.clear()\n                self._visited_targets.clear()\n            elif state is GameState.GAME_OVER:\n                self._deadly.add(edge)\n                self._plan.clear()\n        self._state_visits[current] += 1\n        self._states_seen.add(current)\n\n    def _available_ids(self, frame: FrameData) -> list[int]:\n        available = [int(value) for value in frame.available_actions]\n        if not available:\n            available = [\n                action.value for action in GameAction if action is not GameAction.RESET\n            ]\n        return [value for value in available if value != GameAction.RESET.value]\n\n    def _candidate_keys(self, frame: FrameData, grid: Grid) -> list[ActionKey]:\n        candidates: list[ActionKey] = []\n        for action_id in self._available_ids(frame):\n            if action_id == GameAction.ACTION6.value:\n                candidates.extend(\n                    (action_id, x, y)\n                    for x, y in coordinate_proposals(grid, self._last_grid)\n                )\n            else:\n                candidates.append((action_id, None, None))\n        return candidates\n\n    def _movement_bonus(self, candidate: ActionKey, grid: Grid) -> float:\n        delta = self._moves.get(candidate[0])\n        if delta is None or self._avatar_color is None:\n            return 0.0\n        avatar = self._centroid(grid, self._avatar_color)\n        if avatar is None:\n            return 0.0\n        ax, ay, _ = avatar\n        targets = [\n            (area, color, x, y)\n            for area, color, x, y, _, _ in _components(grid)\n            if color != self._avatar_color\n            and (color, x // 4, y // 4) not in self._visited_targets\n        ]\n        if not targets:\n            return 0.0\n        _, color, tx, ty = min(\n            targets, key=lambda item: (item[0], abs(item[2] - ax) + abs(item[3] - ay))\n        )\n        if abs(tx - ax) + abs(ty - ay) <= 3:\n            return 0.0\n        dx, dy = delta\n        before = abs(tx - ax) + abs(ty - ay)\n        after = abs(tx - (ax + dx)) + abs(ty - (ay + dy))\n        return 3.0 * (before - after)\n\n    def _score(self, current: str, candidate: ActionKey, grid: Grid) -> float:\n        edge = (current, candidate)\n        if edge in self._deadly:\n            return -10_000.0\n        if edge in self._no_ops:\n            return -1_000.0\n        node = self._nodes[current]\n        exploration = 12.0 if candidate not in node.tested else 1.0\n        score = exploration + 8.0 * self._progress_actions[candidate[0]]\n        for changes, tries in self._effect_stats(candidate, grid):\n            score += changes / tries + 1.5 / math.sqrt(tries)\n        score += self._movement_bonus(candidate, grid)\n        return score + self._rng.random() * 0.001\n\n    def _plan_to_frontier(self, start: str) -> bool:\n        parents: dict[str, tuple[str, ActionKey] | None] = {start: None}\n        queue = deque([start])\n        target: str | None = None\n        while queue:\n            state = queue.popleft()\n            node = self._nodes.get(state)\n            if node is None:\n                continue\n            if state != start and any(key not in node.tested for key in node.candidates):\n                target = state\n                break\n            for action, outcome in node.edges.items():\n                if outcome not in parents and (state, action) not in self._deadly:\n                    parents[outcome] = (state, action)\n                    queue.append(outcome)\n        if target is None:\n            return False\n        path: list[tuple[str, ActionKey]] = []\n        cursor = target\n        while parents[cursor] is not None:\n            previous, action = parents[cursor]\n            path.append((previous, action))\n            cursor = previous\n        self._plan = deque(reversed(path))\n        self._telemetry["replans"] += 1\n        return True\n\n    def _materialize(self, key: ActionKey, why: str) -> GameAction:\n        action = GameAction.from_id(key[0])\n        if action is GameAction.ACTION6:\n            action.set_data({"x": int(key[1] or 0), "y": int(key[2] or 0)})\n        action.reasoning = {\n            "controller": "settled-world-model-v2",\n            "why": why,\n            "known_states": len(self._nodes),\n            "movement_model": dict(self._moves),\n        }\n        return action\n\n    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:\n        started = perf_counter()\n        grid = frame_grid(latest_frame)\n        current = state_key(latest_frame)\n        self._observe(current, grid, latest_frame.levels_completed, latest_frame.state)\n\n        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):\n            selected: ActionKey = (GameAction.RESET.value, None, None)\n            action = self._materialize(selected, "start or recover the level")\n        else:\n            node = self._nodes.setdefault(current, Node())\n            if not node.candidates:\n                node.candidates = self._candidate_keys(latest_frame, grid)\n\n            selected = None\n            reason = "select the highest-value untested local transition"\n            if self._plan:\n                expected, planned = self._plan[0]\n                if expected == current:\n                    self._plan.popleft()\n                    selected = planned\n                    reason = "replay a known edge toward an unexplored frontier"\n                else:\n                    self._plan.clear()\n                    self._telemetry["plan_invalidations"] += 1\n\n            untested = [\n                key\n                for key in node.candidates\n                if key not in node.tested and (current, key) not in self._deadly\n            ]\n            if selected is None and untested:\n                selected = max(untested, key=lambda key: self._score(current, key, grid))\n            if selected is None and self._plan_to_frontier(current):\n                _, selected = self._plan.popleft()\n                reason = "return to a known state with untested actions"\n            if selected is None:\n                viable = [\n                    key for key in node.candidates if (current, key) not in self._deadly\n                ]\n                selected = max(\n                    viable or node.candidates,\n                    key=lambda key: self._score(current, key, grid),\n                )\n                reason = "reuse the most promising observed transition"\n            action = self._materialize(selected, reason)\n\n        elapsed_ms = (perf_counter() - started) * 1000.0\n        self._telemetry["actions_selected"] += 1\n        self._selection_reasons[action.reasoning["why"]] += 1\n        self._decision_latency_ms_total += elapsed_ms\n        self._decision_latency_ms_max = max(self._decision_latency_ms_max, elapsed_ms)\n        action.reasoning["telemetry"] = self.telemetry_summary()\n        self._last_state = current\n        self._last_grid = grid\n        self._last_level = latest_frame.levels_completed\n        self._last_action = selected\n        return action\n', encoding='utf-8')
print('BASELINE_SOURCE=lean_effect_graph_explorer; no learned weights')


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('RUN_MODE=competition_rerun: starting the scored ARC agent')
    # Wait for the gateway sidecar to be ready.
    subprocess.run(
        [
            'curl', '--fail', '--retry', '999', '--retry-all-errors',
            '--retry-delay', '5', '--retry-max-time', '600',
            'http://gateway:8001/api/games',
        ],
        check=True,
    )

    # Copy the framework into a writable location.
    framework_source = Path(
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/'
        'ARC-AGI-3-Agents'
    )
    framework_dir = Path('/tmp/ARC-AGI-3-Agents')
    if framework_dir.exists():
        shutil.rmtree(framework_dir)
    shutil.copytree(framework_source, framework_dir)

    # Drop a thin template that imports WorldModelAgent from the bundled
    # `agent` package (reconstructed under /tmp/arc3_agent_pkg earlier).
    # The template is a framework-visible module named `my_agent` for
    # registry compatibility; the real logic lives in the agent package.
    (framework_dir / 'agents/templates/my_agent.py').write_text(
        'from my_agent import MyAgent\n'
    )

    # Register the agent in the framework's registry. We rewrite
    # __init__.py because the upstream version eagerly imports
    # templates with deps we don't ship (langgraph, smolagents, etc.).
    with open(framework_dir / 'agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Point the framework at the gateway sidecar.
    with open(framework_dir / '.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/tmp/arc3_recordings
""")

    # Run only the agent process against the isolated ARC package set.
    # The gateway records every action and emits submission.parquet.
    agent_env = os.environ.copy()
    agent_env['PYTHONPATH'] = (
        '/tmp/arc3_runtime'
        + os.pathsep
        + '/tmp/arc3_baseline'
        + os.pathsep
        + agent_env.get('PYTHONPATH', '')
    )
    agent_env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
    agent_env['MPLBACKEND'] = 'agg'
    agent_env['PYTHONDONTWRITEBYTECODE'] = '1'
    agent_env['PYTHONHASHSEED'] = '0'
    agent_env['PYTHONUNBUFFERED'] = '1'
    subprocess.run(
        [
            sys.executable, '-X', 'frozen_modules=off',
            'main.py', '--agent', 'myagent',
        ],
        cwd=framework_dir,
        env=agent_env,
        check=True,
        timeout=8 * 3600,
    )
else:
    print(
        'RUN_MODE=commit_smoke_test: the private gateway is unavailable; '
        'the scored agent run starts only after Submit to Competition.'
    )


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Save-and-run-all (commit) mode: emit a dummy submission so the
    # commit succeeds. The real submission.parquet is produced by the
    # gateway during competition rerun.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    output_path = '/kaggle/working/submission.parquet'
    submission.to_parquet(output_path, index=False)
    print(
        'COMMIT_ARTIFACT=dummy: wrote schema-valid, unscored '
        f'{output_path}; rows={len(submission)}'
    )
    display(submission.head())


In [ ]:
import os
from pathlib import Path
import pandas as pd

submission_path = Path('/kaggle/working/submission.parquet')
if not submission_path.is_file():
    raise FileNotFoundError(
        f'Expected submission artifact was not created: {submission_path}'
    )

checked_submission = pd.read_parquet(submission_path)
expected_columns = ['row_id', 'game_id', 'end_of_game', 'score']
if list(checked_submission.columns) != expected_columns:
    raise RuntimeError(
        'Invalid submission columns: '
        f'{list(checked_submission.columns)} != {expected_columns}'
    )
if checked_submission.empty:
    raise RuntimeError('Submission artifact contains no rows')
if checked_submission['row_id'].isna().any():
    raise RuntimeError('Submission contains null row_id values')
if checked_submission['row_id'].duplicated().any():
    raise RuntimeError('Submission contains duplicate row_id values')

is_rerun = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
if is_rerun and len(checked_submission) <= 1:
    raise RuntimeError(
        'Competition rerun produced only the one-row commit dummy; '
        'refusing to report a successful scored run'
    )
artifact_kind = 'scored' if is_rerun else 'commit_dummy'
print(
    f'SUBMISSION_VALIDATION=PASS kind={artifact_kind} '
    f'rows={len(checked_submission)} '
    f'games={checked_submission["game_id"].nunique()}'
)
